# Tidal-Current Energy in the Philippines
## A Workshop on the Open-Source Screening & Visualisation Stack

Marine spatial planning (MSP) for tidal in-stream energy — from raw bathymetry and tidal harmonics to an interactive web map.

*Two-phase workflow + high-resolution refinement:*

1. **Screening** — a 2-D depth-averaged shallow-water solver in Python/NumPy.
2. **Web** — a Flask + MapLibre GL JS tool to explore the resource.
3. **Refinement** — TELEMAC-2D on the top hotspots (optional, engine-agnostic).

MIT licensed · https://github.com/anomalyco/tidal-oss

## What we'll cover

| # | Topic |
|---|-------|
| 1 | **The Data** — inputs we ingest and the outputs we publish |
| 2 | **Solution Architecture** — how the pieces fit together |
| 3 | **Processing** — the shallow-water model & power-density maths |
| 4 | **Web Service** — the Flask/MapLibre API and map |
| 5 | **Features** — what the MSP tool can do |

Each section ends with a **live code demo** you can run right here.

# 1 · The Data

Every assessment starts with two kinds of geospatial data:

- **Bathymetry** — how deep the water is (drives friction & funnelling).
- **Tidal forcing** — how the sea surface rises & falls at the open boundaries.

Plus a **land mask** so we never compute currents on dry land.

## Inputs we ingest

| Input | Source | Role |
|-------|--------|------|
| Bathymetry | **GEBCO 2026** NetCDF | Seabed depth *h* (m, positive down) |
| Tidal harmonics | **GOT4.10c** / FES2014 / TPXO9 (or synthetic) | M₂, S₂, K₁, O₁ amplitudes & phases |
| Land mask | **Philippines landmass GeoJSON** (from OSM / GADM) | Mark dry cells |

The model regrids everything onto one uniform **Arakawa C-grid** (~2 km) covering the Philippine bounding box 116–128°E, 4–22°N, and applies the CFL stability condition to pick the time step automatically.

## Outputs we publish

Six canonical files in `output/` (the *output contract* shared by the screening solver and TELEMAC-2D):

| File | Contents |
|------|----------|
| `tidal_power_density.tif` | Time-mean power density **[W/m²]** (primary product) |
| `max_current_speed.tif` | Max depth-averaged speed **[m/s]** |
| `bathymetry.tif` | Bathymetric depth **[m]** |
| `distance_to_coast.tif` | Distance to nearest coast **[km]** |
| `results.nc` | Time series (η, u, v, power) — NetCDF |
| `hotspots.geojson` | Ranked sites with power ≥ 200 W/m² |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _find_power_raster():
    # Locate the canonical screening product whether the notebook is run from
    # the repo root or from docs/notebooks/.
    from pathlib import Path
    candidates = [
        Path('output/tidal_power_density.tif'),
        Path('../../../output/tidal_power_density.tif'),
    ]
    return next((p for p in candidates if p.exists()), None)

def load_power_field():
    # Prefer the real screening output; otherwise synthesise a realistic field.
    try:
        import rasterio
        p = _find_power_raster()
        if p is not None:
            with rasterio.open(p) as src:
                arr = np.ma.masked_invalid(src.read(1).astype(float))
            return np.asarray(arr), str(p), True
    except Exception as exc:
        print('rasterio / real data unavailable:', exc)
    # Synthetic: a noisy shelf with two high-power straits.
    rng = np.random.default_rng(7)
    ny, nx = 120, 200
    x = np.linspace(0, 1, nx); y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)
    base = rng.lognormal(mean=4.0, sigma=0.9, size=(ny, nx)) * 0.15
    s1 = 1600 * np.exp(-((X-0.30)**2)/0.002 - ((Y-0.5)**2)/0.20)
    s2 = 1100 * np.exp(-((X-0.68)**2)/0.0015 - ((Y-0.35)**2)/0.12)
    return np.clip(base + s1 + s2, 0, None), 'synthetic (straits @ x=0.30, 0.68)', False

field, src_label, is_real = load_power_field()
print('source :', src_label)
print(f'min={field.min():.1f}  mean={field.mean():.1f}  max={field.max():.1f} W/m^2')
print(f'P95   ={np.percentile(field,95):.1f} W/m^2')
print('hotspot cells (>=200 W/m^2):', int((field>=200).sum()))

plt.figure(figsize=(9, 4.5))
im = plt.imshow(field, origin='lower', cmap='inferno',
               vmin=0, vmax=max(300, float(np.percentile(field, 99))))
plt.colorbar(im, label='Power density (W/m^2)')
plt.title('Tidal power density - screening output')
plt.xlabel('longitude index'); plt.ylabel('latitude index')
plt.show()

# 2 · Solution Architecture

The design philosophy is **fail fast, fail cheap**: a coarse model finds the obvious hotspots, then only those sites get the expensive high-resolution treatment.

```
Phase A: Screening                          Phase B: Web Visualisation
=================================          ================================

 GEBCO  --│                                          Flask (REST API)
 GOT4.10c -┄-- model.run --│-- results.nc               |
               -│-- tidal_power_density.tif          MapLibre GL JS
               -│-- hotspots.geojson
```

TELEMAC-2D (Phase C) refines the top hotspots and writes the **same six files**, so the web app is engine-agnostic.

## The screening → refinement cascade

```
Coarse Python model (this one)
        |
        v
   Sites with  P_mean > 200 W/m^2   (hotspots.geojson)
        |
        v
   High-resolution TELEMAC-2D unstructured mesh
        |
        v
   Turbine-array CFD / actuator-disk model
        |
        v
   Geophysical / environmental surveys  ->  Pilot deployment
```

Key idea: the screening model is **deliberately conservative** (coarse resolution + depth-averaging under-estimates speed), so any site it flags is almost certainly worth a second look.

# 3 · Processing

At the heart of Phase A is the **2-D depth-averaged shallow-water solver** (`src/model/`). It integrates the SWE on an Arakawa C-grid with a forward-backward time stepper.

**Continuity (mass):**
$$\frac{\partial \eta}{\partial t} + \frac{\partial}{\partial x}(h u) + \frac{\partial}{\partial y}(h v) = 0$$

**Momentum (Newton, linearised):**
$$\frac{\partial u}{\partial t} = -g\,\frac{\partial \eta}{\partial x} - \frac{C_d}{h}\,|u|\,u + f v$$

Pressure gradient accelerates the flow; quadratic bottom friction ($C_d\,|u|u/h$) is the dominant energy sink; Coriolis ($f v$) matters at basin scale.

## From current to power: the ½ρU³ law

A tidal turbine is a kinetic-energy converter. The **instantaneous power per unit swept area** is:

$$P = \tfrac{1}{2}\,\rho\,U^{3} \qquad [\text{W/m}^2]$$

with $\rho = 1025$ kg/m³ (seawater) and $U = \sqrt{u^2+v^2}$.

> The **cubic** dependence is the defining feature: doubling the current speed gives **eight times** the power. A 2.5 m/s site is not 25% better than a 2.0 m/s site — it yields nearly double the power density.

**The funnel effect:** mass conservation $Q = A\,U$ means narrow, deep straits concentrate flow — that is why the Philippine inter-island channels (San Bernardino, Surigao) are hotspots.

## Turning a run into a resource map

- Compute $P(t) = \tfrac12\rho\,(u^2+v^2)^{3/2}$ every time step.
- **Time-average** over a full spring–neap cycle (15 days minimum) so we sample both springs and neaps.
- Flag **hotspots** where $\overline{P} \ge 200$ W/m².
- Enforce the **CFL condition** $\Delta t \le \Delta x / \sqrt{g h}$ (auto-computed; lower `cfl_safety` if you see NaNs).

Outputs are rasterised to a uniform grid (bilinear) for GIS compatibility — the four GeoTIFFs plus `results.nc` and `hotspots.geojson`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Prefer the real 'model' package; fall back to a tiny built-in solver.
try:
    from model.grid import StructuredGrid
    from model.solver import ShallowWaterSolver
    from model.forcing import make_synthetic_tidal_boundary
    from model.utils import speed, power_density
    HAVE_MODEL = True
except Exception as exc:  # pragma: no cover
    HAVE_MODEL = False
    print('model package not importable:', exc, '-> built-in fallback')

def build_grid(nx=60, ny=30, dx=1500.0, depth=60.0):
    if HAVE_MODEL:
        g = StructuredGrid.from_uniform(nx, ny, dx, dx)
        g.h[:] = depth; g.h_u[:] = depth; g.h_v[:] = depth
        g.open_boundary[:, 0] = True          # force the left column
        return g
    g = type('G', (), {})()
    g.nx, g.ny, g.dx, g.dy = nx, ny, dx, dx
    g.h = np.full((ny, nx), depth)
    g.mask = np.ones((ny, nx), bool)
    g.open_boundary = np.zeros((ny, nx), bool); g.open_boundary[:, 0] = True
    return g

def run_model(g, days=2.0, dt=20.0):
    n = int(days*86400/dt)
    if HAVE_MODEL:
        bnd = make_synthetic_tidal_boundary(
            int(g.open_boundary.sum()), amplitude=0.8, constituents=['M2', 'S2'])
        solver = ShallowWaterSolver(g, cd=0.0025)
        solver.set_open_boundary_eta(bnd)
        probe = (g.ny//2, g.nx//2)
        eta_ts = []; spd_ts = []; pd_ts = []
        every = max(1, n//200)
        def cb(s, step):
            if step % every == 0:
                eta_ts.append(s.eta[probe])
                spd_ts.append(float(speed(s.u, s.v)[probe]))
                pd_ts.append(float(power_density(s.u, s.v)[probe]))
            return None
        solver.run(dt, days*86400.0, callback=cb, progress_interval=1e9)
        return np.array(eta_ts), np.array(spd_ts), np.array(pd_ts), solver
    return _fallback_run(g, days, dt)

def _fallback_run(g, days, dt):
    ny, nx, dx, h = g.ny, g.nx, g.dx, g.h
    eta = np.zeros((ny, nx)); u = np.zeros((ny, nx+1)); v = np.zeros((ny+1, nx))
    Cd, g0, w = 0.0025, 9.81, 2*np.pi/(12.42*3600.0)
    n = int(days*86400/dt); probe = (ny//2, nx//2)
    eta_ts = []; spd_ts = []; pd_ts = []; every = max(1, n//200)
    for k in range(n):
        t = k*dt
        eta[:, 0] = 0.8*np.cos(w*t)
        dpx = (eta[:, 1:] - eta[:, :-1])/dx
        sp = np.sqrt((u[:, 1:]**2 + u[:, :-1]**2)/2.0)   # speed at faces 0..nx-1
        u[:, 1:-1] -= dt*(g0*dpx + Cd*sp[:, 1:]*u[:, 1:-1]/h[:, 1:])
        dpy = (eta[1:, :] - eta[:-1, :])/dx
        v[1:-1, :] -= dt*(g0*dpy)
        fx = h*(u[:, 1:] - u[:, :-1]); fy = h*(v[1:, :] - v[:-1, :])
        eta[1:-1, 1:-1] -= dt*(fx[1:-1, 1:-1]/dx + fy[1:-1, 1:-1]/dx)
        if k % every == 0:
            uc = (u[:, :-1] + u[:, 1:])/2.0; vc = (v[:-1, :] + v[1:, :])/2.0
            s = np.sqrt(uc**2 + vc**2)
            eta_ts.append(eta[probe]); spd_ts.append(s[probe])
            pd_ts.append(0.5*1025*s[probe]**3)
    return np.array(eta_ts), np.array(spd_ts), np.array(pd_ts), None

g = build_grid()
eta_ts, spd_ts, pd_ts, solver = run_model(g)
print('solver :', 'repo model package' if HAVE_MODEL else 'built-in fallback')
print(f'mean current speed : {spd_ts.mean():.3f} m/s')
print(f'peak current speed : {spd_ts.max():.3f} m/s')
print(f'mean power density : {pd_ts.mean():.1f} W/m^2')
print(f'peak power density : {pd_ts.max():.1f} W/m^2')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
if solver is not None:
    spd = speed(solver.u, solver.v)
    im = ax[0].imshow(spd, origin='lower', cmap='viridis')
    ax[0].set_title('Current speed |U| (m/s)'); plt.colorbar(im, ax=ax[0])
else:
    ax[0].plot(spd_ts); ax[0].set_title('Current speed at probe (m/s)')
ax[1].plot(eta_ts, label='elevation eta (m)')
ax[1].plot(spd_ts, label='speed |U| (m/s)')
ax[1].set_title('Time series at probe'); ax[1].legend(); ax[1].set_xlabel('sample #')
plt.tight_layout(); plt.show()

# 4 · Web Service

Phase B serves the screening outputs through a **Flask REST API** backed by **MapLibre GL JS** for the interactive map.

- Raster layers (GeoTIFF) are rendered to **colormapped PNG tiles** on the fly.
- All endpoints are cache-friendly and read the same six output files.
- Ships as a Docker image: `docker compose up -d --build` → **http://localhost:8001** (container 5000).

| Endpoint | What it returns |
|----------|-----------------|
| `/api/layers` | Metadata for every layer (bounds, stats, legend) |
| `/api/tiles/{layer}/{z}/{x}/{y}.png` | Colormapped raster tiles |
| `/api/query?lat=&lon=&layer=` | Value at a point |
| `/api/timeseries?lat=&lon=` | Tidal curve from `results.nc` |
| `/api/hotspots?min=&limit=` | Ranked hotspots (GeoJSON) |
| `/api/area_stats` | POST polygon → resource stats |
| `/api/resource` | Filtered-domain totals (area / MW / AEP) |
| `/api/turbines` · `/api/turbine_performance` | Turbine specs & yield |
| `/api/download/{file}` | GeoTIFF / GeoJSON / NetCDF download |


## The map — the screening layers

The interactive map renders each canonical GeoTIFF as a switchable overlay. The screenshots below were captured from a run served over the TELEMAC-2D region outputs (generated in `output/screenshots/`):

![Overview](../../output/screenshots/01_overview.png)

*Overview, then the mean power density, max current speed, bathymetry, and distance-to-coast layers.*

![Power](../../output/screenshots/02_power.png)
![Speed](../../output/screenshots/03_speed.png)
![Depth](../../output/screenshots/04_depth.png)
![Distance](../../output/screenshots/05_distance.png)

In [ ]:
import json
import urllib.request

BASE = 'http://localhost:8001'   # docker compose maps 8001 -> 5000

def get_json(path):
    with urllib.request.urlopen(BASE + path, timeout=3) as r:
        return json.load(r)

try:
    layers = get_json('/api/layers')
    res = get_json('/api/resource?min_power=200')
    print('Connected to', BASE)
    avail = [k for k, v in layers['layers'].items() if v.get('available')]
    print('Available layers:', avail)
    print('Resource (>=200 W/m^2):')
    for k in ('n_cells', 'area_km2', 'mean_power_density',
              'extractable_mw', 'aep_gwh_yr'):
        print(f'  {k:22s}: {res.get(k)}')
except Exception as exc:
    print('Web service not running on', BASE, '->', exc)
    print('Start it with:  docker compose up -d --build')
    print('Then open http://localhost:8001')
    print('Sample /api/resource response:')
    print(json.dumps({
        'n_cells': 1559, 'area_km2': 6120.4,
        'mean_power_density': 412.7, 'extractable_mw': 2526.9,
        'aep_gwh_yr': 22136.1}, indent=2))

# 5 · Features

The MSP tool is more than a map viewer — it supports real site-screening workflows:

- **Site inspector** — click anywhere for point stats, a tidal curve, and turbine yield.
- **Ranked hotspots** — sortable list of the best sites.
- **Export** — download GeoTIFF / GeoJSON / NetCDF.
- **Polygon assessment** — draw a site, get area, MW and AEP.
- **Resource screening** — filter by power & depth → national totals.
- **Measure tool** — distance between points.
- **Turbine performance** — top-10 real turbines, capacity factor & AEP.

## TELEMAC-2D refinement regions

The same map UI serves the high-resolution TELEMAC-2D refinements for the top three hotspot regions (`output/telemac/region-00X/`), rendered with the canonical layers:

![Region 001](../../output/screenshots/06_telemac_region-001.png)
![Region 002](../../output/screenshots/07_telemac_region-002.png)
![Region 003](../../output/screenshots/08_telemac_region-003.png)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from web.turbines import all_turbine_specs, performance
    HAVE_TURB = True
except Exception as exc:
    HAVE_TURB = False
    print('web.turbines not importable:', exc, '-> inline model')

def power_curve_kw(rated_kw, cut_in, cut_out, u_rated, u):
    if u < cut_in or u >= cut_out:
        return 0.0
    if u >= u_rated:
        return rated_kw
    frac = (u - cut_in) / (u_rated - cut_in)
    return rated_kw * frac**3

if HAVE_TURB:
    specs = all_turbine_specs()
    top = specs[0]
    print('Top turbine:', top['name'], f"({top['manufacturer']})")
    print(f"  rated power : {top['rated_power_kw']} kW")
    print(f"  rated speed : {top['rated_speed_mps']} m/s")
    us = np.linspace(0, top['cut_out_mps'] + 0.3, 200)
    p = [power_curve_kw(top['rated_power_kw'], top['cut_in_mps'],
                        top['cut_out_mps'], top['rated_speed_mps'], uu) for uu in us]
    plt.figure(figsize=(7, 4))
    plt.plot(us, p, lw=2)
    for x, lab in [(top['cut_in_mps'], 'cut-in'),
                   (top['rated_speed_mps'], 'rated'),
                   (top['cut_out_mps'], 'cut-out')]:
        plt.axvline(x, ls=':', label=lab)
    plt.xlabel('Current speed U (m/s)'); plt.ylabel('Power (kW)')
    plt.title(f"{top['name']} power curve"); plt.legend(); plt.show()
    if 'spd_ts' in globals():           # reuse the Demo-B speed series
        t_hours = list(np.linspace(0, 2*24, len(spd_ts)))
        perf = performance(top, list(spd_ts), t_hours)
        print(f"  capacity factor : {perf['capacity_factor']*100:.1f} %")
        print(f"  AEP             : {perf['aep_gwh_yr']:.3f} GWh/yr")
        print(f"  % time at rated : {perf['pct_time_at_rated']:.1f} %")
else:
    rated, cin, cout, urated = 2000.0, 0.7, 4.0, 2.2
    us = np.linspace(0, cout + 0.3, 200)
    p = [power_curve_kw(rated, cin, cout, urated, uu) for uu in us]
    plt.figure(figsize=(7, 4)); plt.plot(us, p, lw=2)
    plt.xlabel('Current speed U (m/s)'); plt.ylabel('Power (kW)')
    plt.title('Example turbine power curve (inline)'); plt.show()

# Wrap-up & Quick Start

**Run the whole stack locally:**
```bash
docker compose up -d --build     # web map at http://localhost:8001
python -m src.model.run          # (optional) regenerate outputs
```

**Try the live demos in this notebook:**
- Demo A — visualise the power-density field.
- Demo B — run the shallow-water solver (real `model` pkg or fallback).
- Demo C — query the Flask API.
- Demo D — plot a turbine power curve & yield.

**Docs:** `docs/README.md`, `docs/concepts/MODEL.md`, `docs/architecture/WORKFLOW.md`, `docs/engines/TELEMAC.md`, `docs/notebooks/EXPLAINER.ipynb`.

MIT licensed · contributions welcome.